# DFXM Identify Cluster Sweep — Showcase

This notebook presents headline counts and example images from a completed
cluster-scale `dfxm-identify` sweep generated by `lsf/identify_sweep_array.bsub`.

**Before running:** ensure `presentation_assets/` (produced by
`tools/sweep_summary_identify.py`) is placed next to this notebook, or run
jupyter from the directory that contains `presentation_assets/`.

In [ ]:
import json
from pathlib import Path


# Locate presentation_assets/: look in CWD first, then walk up to the repo root.
# This lets the notebook be executed from the repo root (nbconvert default) or
# from examples/cluster_showcase/ when presentation_assets/ lives at the root.
def _find_assets() -> Path:
    for candidate in [
        Path("presentation_assets"),
        Path("../../presentation_assets"),
        Path("../presentation_assets"),
    ]:
        resolved = candidate.resolve()
        if (resolved / "summary.json").exists():
            return resolved
    raise FileNotFoundError(
        "Could not find presentation_assets/summary.json.\n"
        "Run: python tools/sweep_summary_identify.py\n"
        "Then place presentation_assets/ next to this notebook."
    )


assets = _find_assets()
summary_path = assets / "summary.json"

with summary_path.open(encoding="utf-8") as fh:
    summary = json.load(fh)

print("=== Sweep summary ===")
print(f"  Configs processed : {summary['n_configs']}")
print(f"  Total images       : {summary['n_images']}")
print(f"  Dislocations/image : {summary['dislocations_per_frame']}")
print(f"  Total dislocations : {summary['n_images'] * summary['dislocations_per_frame']}")
print()
print("Slip-plane counts:")
for k, v in sorted(summary.get("slip_plane_counts", {}).items(), key=lambda x: -x[1]):
    print(f"  {k:>20} : {v}")

In [ ]:
import matplotlib.pyplot as plt

frames_dir = assets / "frames"
frame_paths = sorted(frames_dir.glob("frame_*.png"))[:8]

if not frame_paths:
    print("No frames found in", frames_dir)
else:
    n = len(frame_paths)
    fig, axes = plt.subplots(2, 4, figsize=(13, 6))
    fig.suptitle("Example detector frames from the identify sweep", fontsize=13)
    for i, ax in enumerate(axes.ravel()):
        if i < n:
            img = plt.imread(str(frame_paths[i]))
            ax.imshow(img, aspect="auto")
            ax.set_title(frame_paths[i].stem, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np

rotations_path = assets / "rotations.npy"
if not rotations_path.exists():
    print("rotations.npy not found — skipping label-distribution plots.")
else:
    rotations = np.load(str(rotations_path))

    slip_counts = summary.get("slip_plane_counts", {})

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Rotation-degree histogram.
    axes[0].hist(rotations, bins=36, color="steelblue", edgecolor="white", linewidth=0.4)
    axes[0].set_title("Distribution of rotation_deg", fontsize=11)
    axes[0].set_xlabel("rotation_deg")
    axes[0].set_ylabel("count")

    # Slip-plane counts bar chart.
    if slip_counts:
        labels = list(slip_counts.keys())
        values = list(slip_counts.values())
        axes[1].bar(range(len(labels)), values, color="coral", edgecolor="white", linewidth=0.4)
        axes[1].set_xticks(range(len(labels)))
        axes[1].set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
        axes[1].set_title("Slip-plane normal counts", fontsize=11)
        axes[1].set_ylabel("count")
    else:
        axes[1].set_visible(False)

    plt.tight_layout()
    plt.show()